# Sentence Extraction from PMC articles

In [ ]:
#Download 10 sample articles for demonstration
!python3 download_pmc.py

In [ ]:
import lexas.sentence
import lexas.relation_extraction
import torch

# Define the device to be used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Step 1: Extracting result sections from the articles
# The 'extract_results' function is used to extract sections of results from articles.
lexas.sentence.extract_results(
    article_dir="./articles/",
    output_file="./data/result_sections.txt",
)

In [ ]:
# Step 2: Masking gene terms and experiments
# The 'mask_gene_experiment' function is used to replace gene terms and experiments with MASK tokens in the text.
lexas.sentence.mask_gene_experiment(
    input_file_path="./data/result_sections.txt",
    output_file_path="./data/masked_sentences.txt",
)

In [ ]:
# Step 3: Relation extraction using BioBERT
# The 'predict' function is used to predict relations using BioBERT model on the masked sentences.
lexas.relation_extraction.predict(
    device=device,
    input_filepath="./data/masked_sentences.txt",
    output_filepath="./data/masked_sentences_bert.txt",
)

# Prediction model for genes

In [ ]:
import lexas.prediction
import xgboost as xgb
from sklearn.model_selection import train_test_split

In [ ]:
# Step 1: Extracting context from experiments
# The 'extract_context_from_experiments' function processes the BioBERT predictions
# and extracts the context in which each experiment mention was made.
lexas.prediction.extract_context_from_experiments(
    input_file="./data/masked_sentences_bert.txt",
    output_file="./data/experiments_for_xgboost.csv"
)

In [ ]:
# Step 2: Loading feature data
# The 'feature_load' function is used to load feature data from various resources.
lexas.prediction.feature_load()

In [ ]:
# Step 3: Selecting features
# The 'select_features' function is used to select the features to be used in the model.
cat_use = ['Chromosome', 'GO', 'MGI', 'HPO', 'OMIM', 'TF', 'iRefIndex', 'Localization', 'WebSter']
num_use = ['Tissue_expression', 'Cancer_expression', 'DepMap', 'Word2Vec']
plus = ["String","Funcoup","GOSemSim"]
feature_list, gene_cat, gene_num = lexas.prediction.select_features(cat_use, num_use)

# Print feature information
print("List of features: ", feature_list[:10])
print("\nFeatures assigned to a gene: ", gene_cat["CDK1"][:10])
print("\nNumerical features assigned to a gene: ", gene_num.keys())

In [ ]:
# Step 4: Constructing the CSR matrix
# The 'construct_csr_matrix' function is used to transform the data into a format that can be processed by the XGBoost model.
path_to_csv="./data/experiments_for_xgboost.csv"
posi_tuple, nega_tuple = lexas.prediction.generate_experiment_tuples(path_to_csv, 1990, 2018, negative_sampling=3)
X, y = lexas.prediction.construct_csr_matrix(posi_tuple, nega_tuple, gene_cat, feature_list, gene_num, additional_features=plus)

In [ ]:
# Step 5: Train-Test split
# The train_test_split function is used to split the data into training and testing sets for model training and evaluation.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Step 6: Model Training
# The XGBClassifier is used to train a model on the data. The model is then saved using the joblib library.
model = xgb.XGBClassifier(
    objective= "binary:logistic",
    alpha=1e-3, 
    min_child_weight=3,
    max_depth=10,
    n_estimators=40000,
    n_jobs=-1,
    eta=0.03
)
model.fit(X_train, y_train, early_stopping_rounds=3, eval_set=[[X_test, y_test]])

# Save the model to a file
from joblib import dump
model_path = "./model/xgboost.joblib"
dump(model, model_path)

# Gene prediction for the next experiment

In [ ]:
import os
import lexas.prediction
import pandas as pd
from joblib import load

# Step 1: Load model
model_name = "xgboost"

model_filepath = f"./model/{model_name}.joblib"

if os.path.exists(model_filepath):
    model = load(model_filepath)
else:
    raise Exception("Model file does not exist: " + model_filepath)

In [ ]:
# Step 2: Generate scores
# The 'generate_scores' function is used to score all genes in relation to the query using the XGBoost model.
query = "CEP152"
scores = lexas.prediction.generate_scores(query, model_name, model, gene_cat, feature_list, gene_num, additional_features=plus)

In [ ]:
# Step 3: Display result
# The result is displayed as a DataFrame sorted by the XGBoost score in descending order.
output_dir = f"./result/{model_name}"
os.makedirs(output_dir, exist_ok=True)

df = pd.DataFrame(scores)
df.to_csv(os.path.join(output_dir,f"{query}.csv"),index=False)
df.sort_values(model_name, ascending=False)[:10]